In [ ]:
from meco import MeCO
from meco_opt import MeCO_Opt
from basic_optimizer import Env

In [ ]:
import constraint_optimization_problem as cop
from problem_utils import CECConstrainedProblem
agent = MeCO(n_state=10, n_action=11)
opt = MeCO_Opt(init_pop_size=50, min_pop_size=50, maxfes=2500)

total_problems = [
    cop.C01, cop.C02, cop.C03, cop.C04, cop.C05, cop.C06, cop.C07,
    cop.C08, cop.C09, cop.C10, cop.C11, cop.C12, cop.C13, cop.C14,
    cop.C15, cop.C16, cop.C17, cop.C18, cop.C19, cop.C20, cop.C21,
    cop.C22, cop.C23, cop.C24, cop.C25, cop.C26, cop.C27, cop.C28,
]


In [ ]:
target_problem = 0 # 训练其余的， 测试这个
epochs = 50
dim = 50

train_probelms = []
for i, problem in enumerate(total_problems):
    if i != target_problem:
        train_probelms.append(problem)

In [ ]:
seed = 42
import numpy as np
import torch
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

train_rng = np.random.RandomState(seed)

best_return = None
best_model_state = None
for epoch in range(epochs):
    # 打乱训练问题的顺序
    train_rng.shuffle(train_probelms)
    for problem in train_probelms:
        cec_problem = CECConstrainedProblem(problem, dim)
        maxfes = 50 * dim
        opt = MeCO_Opt(init_pop_size=50, min_pop_size=50, maxfes=maxfes)
        env = Env(cec_problem, opt)
        train_seed = train_rng.randint(0, 10000)
        episode_return = agent.train_episode(env, seed=train_seed)
        # print(f"Epoch {epoch+1}/{epochs}, Problem: {problem}, Return: {episode_return:.4f}")
    
    # 每隔5个epoch 在训练集上推理一次
    if (epoch + 1) % 5 == 0:
        print(f"--- Evaluation after Epoch {epoch+1} ---")
        RETURN = 0
        for problem in train_probelms:
            cec_problem = CECConstrainedProblem(problem, dim)
            maxfes = 50 * dim
            opt = MeCO_Opt(init_pop_size=50, min_pop_size=50, maxfes=maxfes)
            env = Env(cec_problem, opt)
            eval_seed = 42
            results = agent.run_episode(env, seed=eval_seed)
            RETURN += results['Return']
            # print(f"Evaluation - Problem: {problem}, Return: {results['Return']:.4f}")
        print(f"Average Return on Training Problems: {RETURN / len(train_probelms):.4f}")
        if best_return is None or RETURN > best_return:
            best_return = RETURN
            # 保存 模型
            best_model_state = agent.ddqn.state_dict()



In [ ]:
# 在测试集上评估 跑10次
print(f"--- Final Evaluation on Target Problem ---")
seed_list = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
RESULTS = []
for _ in range(10):
    target_cec_problem = CECConstrainedProblem(total_problems[target_problem], dim)
    maxfes = 50 * dim
    seed = seed_list[_]
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    opt = MeCO_Opt(init_pop_size=50, min_pop_size=50, maxfes=maxfes)
    env = Env(target_cec_problem, opt)
    agent.ddqn.load_state_dict(best_model_state)
    results = agent.run_episode(env, seed=seed)
    RESULTS.append(results)
    

In [ ]:
# 绘制结果
import matplotlib.pyplot as plt
VALUES = []
for result in RESULTS:
    costs_gen_list = result['costs_gen_list']
    penalty_gen_list = result['penalties_gen_list']
    Gen = len(costs_gen_list)
    
    run_values = []
    for i in range(Gen):
        values = costs_gen_list[i] + np.sum(penalty_gen_list[i], axis=1)
        run_values.append(np.min(values))
    VALUES.append(run_values)
VALUES = np.array(VALUES)
print(VALUES.shape)


In [ ]:
# 绘制折线和误差带

mean_values = np.mean(VALUES, axis=0)
std_values = np.std(VALUES, axis=0)
plt.figure(figsize=(10, 6))
X = range(1, len(mean_values) + 1)
plt.plot(X, mean_values, label='Mean Performance', color='blue')
plt.fill_between(X, mean_values - std_values, mean_values + std_values, color='blue', alpha=0.2, label='Std Dev')
plt.title('Mean Performance with Standard Deviation')
plt.xlabel('Generation')
plt.ylabel('performance')
plt.legend()
plt.grid()
plt.show()
